# AWF Ladder Benchmark

**Compress any LLM at multiple ratios and benchmark each rung.**

Ladder: Baseline → 2× → 4× → 8× → 10× → 16× → 24×

Metrics per rung:
- HumanEval pass rate (code generation)
- Perplexity (language quality)
- Tokens/sec (inference speed)
- Peak RAM / VRAM (memory usage)
- Model loading time
- File size on disk

## Memory Safe
Compresses layer-by-layer, frees original weights immediately.
Never holds full uncompressed + compressed model simultaneously.
Safe for Colab (won't crash runtime).

## Supported Models
- `distilgpt2` (82M, safe for testing)
- `Qwen/Qwen2-1.5B-Instruct` (1.5B, recommended)
- `microsoft/Phi-3-mini-4k-instruct` (3.8B)
- `THUDM/glm-4-9b-chat` (9B, needs Colab A100 or high-RAM)
- `deepseek-ai/deepseek-llm-7b-chat` (7B)
- `Qwen/Qwen2-7B-Instruct` (7B)
- `mistralai/Mistral-7B-Instruct-v0.3` (7B, needs HF token)
- `meta-llama/Llama-3.1-8B-Instruct` (8B, needs HF token)

In [ ]:
# @title Setup
!git clone https://github.com/Deexv/AWF.git 2>/dev/null || true
%cd AWF
!pip install -q -r requirements.txt transformers

import torch, sys, os, gc
sys.path.insert(0, '.')
sys.path.insert(0, 'scripts')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Step 1: Choose Model and Rungs

**IMPORTANT: Choose a model that fits your Colab VRAM.**
- Colab Free T4: 16GB VRAM → max 7B model (fp32) or 13B (fp16)
- Colab Pro A100: 40GB VRAM → max 30B model (fp16)

**For 30B models (GLM-4-9B is 9B, not 30B):**
- GLM-4-9B-Chat: 9B params, ~36GB fp32, ~18GB fp16 → needs A100
- DeepSeek 7B: 7B params, ~28GB fp32, ~14GB fp16 → works on T4 (fp16)

In [ ]:
# @title Configuration
model_name = 'Qwen/Qwen2-1.5B-Instruct'  # @param ['distilgpt2', 'Qwen/Qwen2-1.5B-Instruct', 'microsoft/Phi-3-mini-4k-instruct', 'THUDM/glm-4-9b-chat', 'deepseek-ai/deepseek-llm-7b-chat', 'Qwen/Qwen2-7B-Instruct', 'mistralai/Mistral-7B-Instruct-v0.3', 'meta-llama/Llama-3.1-8B-Instruct']
rungs = '1,2,4,8'  # @param {type:'string'}
use_fp16 = True  # @param {type:'boolean'}

print(f'Model: {model_name}')
print(f'Rungs: {rungs}')
print(f'Use fp16: {use_fp16} (saves VRAM for big models)')

## Step 2: Run the Ladder Benchmark

This compresses the model at each ratio and measures all metrics.

**Memory-safe**: compresses layer-by-layer, frees memory between layers.

In [ ]:
# @title Run AWF Ladder
import sys, os, time, json, gc, math, traceback
sys.path.insert(0, '.')
sys.path.insert(0, 'scripts')
import torch
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

from transformers import AutoModelForCausalLM, AutoTokenizer
from awf_ladder import (
    compress_layer_by_layer, measure_perplexity, measure_tokens_per_sec,
    measure_humaneval, generate_sample, get_peak_memory, reset_memory_stats
)

# Parse rungs
rung_mults = [float(x) for x in rungs.split(',')]
keep_ratios = [1.0 / m for m in rung_mults]

print(f'Device: {device}')
print(f'Model: {model_name}')
print(f'Rungs: {rung_mults}')
print(f'Keep ratios: {keep_ratios}')

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

all_results = []
base_model = None

for mult, kr in zip(rung_mults, keep_ratios):
    rung_name = 'baseline' if mult == 1.0 else f'{mult:.0f}x'
    print(f'\n{"="*60}')
    print(f'  RUNG: {rung_name} (keep_ratio={kr:.3f})')
    print(f'{"="*60}')

    try:
        load_start = time.time()

        if base_model is None:
            # First rung: load fresh model
            dtype = torch.float16 if use_fp16 and device == 'cuda' else torch.float32
            print(f'  Loading {model_name} (dtype={dtype})...')
            model = AutoModelForCausalLM.from_pretrained(
                model_name, torch_dtype=dtype, trust_remote_code=True
            ).to(device)
            model.eval()
            base_model = model
        else:
            model = base_model

        # Compress if not baseline
        if kr < 0.99:
            print(f'  Compressing (layer-by-layer, memory-safe)...')
            compressed_data, ratio, n_layers = compress_layer_by_layer(model, kr, device)
        else:
            compressed_data = None
            ratio = 1.0
            n_layers = 0

        load_time = time.time() - load_start

        # Metrics
        n_params = sum(p.numel() for p in model.parameters())
        model_size_mb = n_params * (2 if use_fp16 and device == 'cuda' else 4) / 1024 / 1024
        peak_ram = get_peak_memory(device)
        reset_memory_stats(device)

        print(f'  Measuring perplexity...')
        ppl, loss = measure_perplexity(model, tokenizer, device)

        print(f'  Measuring tokens/sec...')
        tps = measure_tokens_per_sec(model, tokenizer, device)

        print(f'  Measuring HumanEval...')
        he_pass = measure_humaneval(model, tokenizer, device)

        peak_inf = get_peak_memory(device)

        sample = generate_sample(model, tokenizer, device,
                                  'Write a Python function that adds two numbers:', max_tokens=60)

        # File size (compressed)
        if compressed_data is not None:
            save_name = model_name.replace('/', '_') + f'_rung_{rung_name}.pt'
            save_path = f'checkpoints/{save_name}'
            torch.save({'model_name': model_name, 'compressed_weights': compressed_data,
                        'keep_ratio': kr}, save_path)
            file_size_mb = os.path.getsize(save_path) / 1024 / 1024
            del compressed_data
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        else:
            file_size_mb = model_size_mb

        results = {
            'rung': rung_name, 'keep_ratio': kr,
            'n_params': n_params, 'model_size_mb': round(model_size_mb, 1),
            'file_size_mb': round(file_size_mb, 1),
            'compression': round(1/max(ratio, 0.01), 1),
            'load_time_sec': round(load_time, 1),
            'perplexity': round(ppl, 2), 'loss': round(loss, 4),
            'tokens_per_sec': round(tps, 1),
            'humaneval_pass_rate': round(he_pass, 3),
            'peak_ram_mb': round(peak_ram, 1),
            'peak_vram_mb': round(peak_inf, 1) if device == 'cuda' else 0,
            'sample': sample[:150],
        }
        all_results.append(results)

        print(f'\n  File: {file_size_mb:.1f} MB | PPL: {ppl:.2f} | Tok/s: {tps:.1f} | HumanEval: {he_pass*100:.0f}%')

    except Exception as e:
        print(f'\n  ERROR at {rung_name}: {e}')
        traceback.print_exc()
        all_results.append({'rung': rung_name, 'error': str(e)[:200]})
        break

print(f'\n{"="*70}')
print(f'LADDER COMPLETE')
print(f'{"="*70}')

## Step 3: View Results Table

In [ ]:
# @title Results Summary
print(f'Model: {model_name}')
print(f'{"Rung":<8} {"File MB":>8} {"Compr":>6} {"PPL":>8} {"Tok/s":>7} {"HumanEval":>10} {"RAM MB":>8} {"Load s":>7}')
print(f'{"-"*65}')
for r in all_results:
    if 'error' in r:
        print(f'{r["rung"]:<8} ERROR: {r["error"][:50]}')
    else:
        print(f'{r["rung"]:<8} {r["file_size_mb"]:>8.1f} {r["compression"]:>5.1f}x {r["perplexity"]:>8.2f} '
              f'{r["tokens_per_sec"]:>7.1f} {r["humaneval_pass_rate"]*100:>9.0f}% {r["peak_ram_mb"]:>8.0f} '
              f'{r["load_time_sec"]:>7.1f}')

In [ ]:
# @title Plot compression vs quality tradeoff
import matplotlib.pyplot as plt

valid = [r for r in all_results if 'error' not in r]
if valid:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    rungs = [r['rung'] for r in valid]
    sizes = [r['file_size_mb'] for r in valid]
    ppls = [r['perplexity'] for r in valid]
    tps = [r['tokens_per_sec'] for r in valid]
    he = [r['humaneval_pass_rate'] * 100 for r in valid]

    axes[0, 0].bar(rungs, sizes, color='steelblue')
    axes[0, 0].set_title('File Size (MB)')
    axes[0, 0].set_ylabel('MB')

    axes[0, 1].bar(rungs, ppls, color='coral')
    axes[0, 1].set_title('Perplexity (lower = better)')
    axes[0, 1].set_ylabel('PPL')
    if max(ppls) > 1000:
        axes[0, 1].set_yscale('log')

    axes[1, 0].bar(rungs, tps, color='green')
    axes[1, 0].set_title('Tokens/sec (higher = faster)')
    axes[1, 0].set_ylabel('tok/s')

    axes[1, 1].bar(rungs, he, color='purple')
    axes[1, 1].set_title('HumanEval Pass Rate (%)')
    axes[1, 1].set_ylabel('%')

    plt.suptitle(f'AWF Ladder: {model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('benchmarks/ladder_plot.png', dpi=150)
    plt.show()
    print('Saved to benchmarks/ladder_plot.png')

## Step 4: View Sample Outputs Per Rung

See how text quality degrades at each compression level.

In [ ]:
for r in all_results:
    if 'sample' in r:
        print(f'\n--- {r["rung"]} (PPL={r.get("perplexity", "?")}) ---')
        print(r['sample'])

## Step 5: Save Results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/AWF

# Save results JSON
import json
results_path = f'benchmarks/ladder_{model_name.replace("/", "_")}.json'
with open(results_path, 'w') as f:
    json.dump({'model': model_name, 'results': all_results}, f, indent=2)
!cp {results_path} /content/drive/MyDrive/AWF/
!cp benchmarks/ladder_plot.png /content/drive/MyDrive/AWF/ 2>/dev/null || true
print(f'Saved to Google Drive: /content/drive/MyDrive/AWF/')
print(f'  - {os.path.basename(results_path)}')
print(f'  - ladder_plot.png')

## Running Big Models (7B-30B)

### GLM-4-9B-Chat (9B params)
```python
model_name = 'THUDM/glm-4-9b-chat'
rungs = '1,2,4'
use_fp16 = True  # MUST use fp16 for 9B model on Colab T4
```

### DeepSeek 7B Chat
```python
model_name = 'deepseek-ai/deepseek-llm-7b-chat'
rungs = '1,2,4,8'
use_fp16 = True
```

### Qwen2 7B Instruct
```python
model_name = 'Qwen/Qwen2-7B-Instruct'
rungs = '1,2,4,8'
use_fp16 = True
```

### Llama 3.1 8B Instruct (needs HF token)
```python
!huggingface-cli login  # paste your token
model_name = 'meta-llama/Llama-3.1-8B-Instruct'
rungs = '1,2,4'
use_fp16 = True
```

### Memory Tips
- **Colab Free T4 (16GB VRAM)**: max 7B model in fp16, max 3B in fp32
- **Colab Pro A100 (40GB VRAM)**: max 20B in fp16, max 9B in fp32
- **If it crashes**: use fewer rungs ('1,2' instead of '1,2,4,8')
- **If OOM during compression**: the layer-by-layer method frees memory, but loading the original model still needs RAM. Use fp16.